<a id="top"></a> 
# Text Analysis Assessment Harry Potter Reviews

# Table of Contents 
1. [Business Understanding](#business-understanding) 
2. [Data Understanding](#data-understanding) 
3. [Baseline-Models](#baseline-models) 
4. [Data Preparation](#data-preparation) 
5. [Hyperparameter Tuning](#hyperparameter-tuning) 
6. [Evaluation](#evaluation) 
7. [References](#references) 
8. [Appendix](#appendix) 
<h2 id ="business-understanding">Business Understanding</h2> 
<a href="#top">Back to top</a>

<h2 id = "introduction">Introduction</h2>
<a href="#top">Back to top</a>
This is a text mining analysis of a Harry Potter Review dataset using Python. The notebook is structured in line with the CRISP-DM methodology. The main aim of the project is to classify unstructured review text into two sentiment categories. The written review comments are transformed into numerical feature representations and evaluated using multiple text mining techniques.

<h2 id="business-understanding">Business Understanding</h2>
<a href="#top">Back to top</a>

<h3 id="business-objective">Business Objective</h3>
<a href="#top">Back to top</a>

The objective of this project is to mine unstructured text data and build a text classification pipeline capable of telling the difference between positive and negative review sentiment.

<h3 id="project-aim">Project Aim</h3>
<a href="#top">Back to top</a>
The projects aim is to compare three baseline vectorisation techniques, apply a suitable clasification algorithm, explore the text through statistics and visualisations and iteratively improve performance through preprocessing, feature selection and hyperamerer tuning.

<h3 id="success-criteria">Success Criteria</h3>
<a href="#top">Back to top</a>
As sucessful outcome is a final text mining pipeline that produces strong and consistant cross-validation performance,that is supported by findings from data understanding and preparation.


In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
import warnings

from sklearn.svm import SVC
from sklearn.metrics import adjusted_rand_score, fowlkes_mallows_score
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score

warnings.filterwarnings("ignore")

import text_mining_utils as tmu

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('omw-1.4')

<h2 id="data-understanding">Data Understanding</h2>
<a href="#top">Back to top</a>

<h3 id="dataset-description">Dataset Description</h3>
<a href="#top">Back to top</a>
The dataset used in this analysis is a CSV file containing Harry Potter review data. The main text field is the (comment) column, which contains the unstructured review text. The (rating) column is used to derive binary class labels for the classification task.

<h3 id="initial-inspection">Initial Inspection</h3>
<a href="#top">Back to top</a>

The dataset is first loaded and inspected to identify its dimensions, available attributes, missing values, and general suitability for text mining.


In [ ]:
data = pd.read_csv('HPotterReviews.csv')
print("Shape:", data.shape)

In [ ]:

print(data.head(5))
print(data.tail(10))
print(data.info)


In [ ]:
print("Shape:", data.shape)
print("\nColumns:")
print(data.columns.tolist())

In [ ]:
data.isnull().sum()

In [ ]:
duplicate_count = data.duplicated(subset=['comment']).sum()
print("Duplicate comments:", duplicate_count)

In [ ]:
data = data.dropna(subset=['comment', 'rating'])
data = data.drop_duplicates(subset=['comment']).reset_index(drop=True)

print("Shape after removing missing values and duplicates:", data.shape)

<h3 id="label-creation">Label Creation</h3>
<a href="#top">Back to top</a>

To simplify the classification task, I created a binary sentiment label from the review rating. Reviews with a rating below 3.5 are labelled as negative, while reviews with a rating of 3.5 or above are labelled as positive. 

In [ ]:
def make_label(rating):
    if rating >= 3.5:
        return "positive"
    else:
        return "negative"

data["label"] = data["rating"].apply(make_label)

data[["rating", "label"]].head()

In [ ]:
data["label"].value_counts()

In [ ]:
label_counts = data["label"].value_counts()

plt.figure(figsize=(6,4))
plt.bar(label_counts.index, label_counts.values)
plt.title("Class Distribution")
plt.xlabel("Label")
plt.ylabel("Number of Reviews")
plt.show()

<h3 id="word-token-statistics">Word/Token Statistics</h3>
<a href="#top">Back to top</a>

Two types of token statistics are derived for each class in order to better understand the characteristics of the text. These statistics help identify differences in review length, vocabulary usage, and repitition across the positive and negative categories.

In [ ]:
positive_docs = data.loc[data["label"] == "positive", "comment"].astype(str).tolist()
negative_docs = data.loc[data["label"] == "negative", "comment"].astype(str).tolist()

print("Positive documents:", len(positive_docs))
print("Negative documents:", len(negative_docs))

In [ ]:
def get_token_stats(docs):
    tokenised_docs = [nltk.word_tokenize(doc) for doc in docs]
    all_tokens = [token for doc in tokenised_docs for token in doc]
    vocab = set(all_tokens)
    avg_doc_length = np.mean([len(doc) for doc in tokenised_docs])
    lexical_diversity = len(vocab) / len(all_tokens) if len(all_tokens) > 0 else 0
    
    return {
        "documents": len(docs),
        "total_tokens": len(all_tokens),
        "vocabulary_size": len(vocab),
        "average_document_length": round(avg_doc_length, 2),
        "lexical_diversity": round(lexical_diversity, 4)
    }

In [ ]:
positive_stats = get_token_stats(positive_docs)
negative_stats = get_token_stats(negative_docs)

stats_df = pd.DataFrame([positive_stats, negative_stats], index=["positive", "negative"])
stats_df

The first statistic examined is the average document length, which indicated whether one class tends to contain longer reviews than the other. The second statistic is lexical diversity, this shows how varied the vocabulary is within each class. Together, these measures help indicate whether one category is more repetitive, more descriptive, or more linguistically varied than the other. 

<h3 id="frequent-terms">Frequent Terms by Category</h3>
<a href="#top">Back to top</a>

To identify the most common tokens in each category, the combined text for each class is examined. This helps highlight repeated concepts, possible stop words, and candidate terms for later preprocessing.

In [ ]:
#join the documents into a single string for each class
positive_text = " ".join(positive_docs)
negative_text = " ".join(negative_docs)

In [ ]:
# Print the 15 most frequent words for each class
tmu.print_n_mostFrequent("positive", positive_text, 15)
print()
tmu.print_n_mostFrequent("negative", negative_text, 15)

In [ ]:
from collections import Counter

def most_frequent_df(text, n=15):
    tokens = nltk.word_tokenize(text)
    counter = Counter(tokens)
    top_n = counter.most_common(n)
    df = pd.DataFrame(top_n, columns=["token", "count"])
    df["relative_frequency"] = df["count"] / len(tokens)
    return df

positive_freq_df = most_frequent_df(positive_text, 15)
negative_freq_df = most_frequent_df(negative_text, 15)

positive_freq_df

In [ ]:
#Display the negative version too
negative_freq_df

<h3 id="visual-analysis">Visual Analysis</h3>
<a href="#top">Back to top</a>

Two visualisation technoques are used to examine the text by class:
1. bar charts displaying the most frquent terms and 
2. word clouds.
These help identify common concepts, repeated vocabulary, possible stop words, and word variations that may influence later preparation steps.

In [ ]:
#Bar chart for positive review
plt.figure(figsize=(10,5))#Bar chart for positive review
plt.bar(positive_freq_df["token"], positive_freq_df["count"])
plt.title("Top 15 Most Frequent Tokens in Positive Reviews")
plt.xlabel("Token")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
#Bar chart for negative review
plt.figure(figsize=(10,5))
plt.bar(negative_freq_df["token"], negative_freq_df["count"])
plt.title("Top 15 Most Frequent Tokens in Negative Reviews")
plt.xlabel("Token")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
#Word clouds for positive reviews
tmu.generate_cloud(positive_text, "Positive Reviews", bg_colour="white", min_font=10)
plt.show()

In [ ]:
#Word clouds for positive reviews
tmu.generate_cloud(negative_text, "Negative Reviews", bg_colour="white", min_font=10)
plt.show()

The bar charts and word clouds are used to identify the most common terms in each category. These outputs will be examined to detremine whether there are obvious repeated function words, possible stop words, or variations of similar concepts that should be handled during preprocessing. They also help identify terms that may later become strong predictive features.